In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
import plotly.express as px
from tqdm import tqdm


In [ ]:
pm_india_df = pd.read_csv("PM-India-Parallel-Corpus/combined_translations.csv")

In [ ]:
pm_india_df.head(1)

### Loading: AIBharat/IndicBERT model


In [ ]:
# check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
model_name = "ai4bharat/indic-bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

In [ ]:
def get_indicbert_embedding(sentence):
    if not isinstance(sentence, str) or sentence.strip() == "":
        return np.nan  # Handle empty or NaN values

    tokens = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True, max_length=521).to(device)

    with torch.no_grad():
        outputs = model(**tokens)
    # Mean pooling : averages the token embeddings along the sequence length axis (dimension 1), resulting in a single vector for the whole sentence (mean pooling).
    embeddings = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return embeddings

In [ ]:
# Create a new DataFrame to store embeddings
pm_ind_embeddings_df = pd.DataFrame(index=pm_india_df.index, columns=pm_india_df.columns)

# Generate embeddings with progress tracking
for col in tqdm(pm_india_df.columns, desc="Generating Embeddings"):
    pm_ind_embeddings_df[col] = pm_india_df[col].apply(get_indicbert_embedding)



### embeddings dimensions - 768

In [ ]:
pm_ind_embeddings_df.head(1)

In [ ]:
pm_ind_embeddings_df.iloc[0].apply(lambda x: x.shape if isinstance(x, np.ndarray) else type(x))

### t-SNE dimensionality reduction

In [ ]:
def prepare_tsne_data(df):
    data = []
    labels = []
    row_indices = []

    for row_idx, row in df.iterrows():
        for lang in df.columns:
            embedding = row[lang]
            data.append(embedding)
            labels.append(lang)
            row_indices.append(row_idx)

    return np.array(data), labels, row_indices


- n_components=2
- perplexity=5
- max_iter=2000,
- learning_rate=100,
- metric="cosine",
- random_state=42




In [ ]:
# Prepare
all_embeddings, languages, row_ids = prepare_tsne_data(pm_ind_embeddings_df)

tsne = TSNE(
    n_components=2,
    perplexity=5,
    max_iter=2000,
    learning_rate=100,
    metric="cosine",
    random_state=42
)

tsne_result = tsne.fit_transform(all_embeddings)

# Make DataFrame
tsne_df = pd.DataFrame({
    'x': tsne_result[:, 0],
    'y': tsne_result[:, 1],
    'Language': languages,
    'Row': row_ids
})


In [ ]:

fig = px.scatter(
    tsne_df,
    x='x',
    y='y',
    color='Row',
    hover_data=['Language', 'Row'],
    title="t-SNE of Multilingual Sentence Embeddings (Colored by Row)",

)
fig.update_traces(marker=dict(size=6, opacity=0.8))
fig.show()

In [ ]:

fig = px.scatter(
    tsne_df,
    x='x',
    y='y',
    color='Language',
    hover_data=['Language', 'Row'],
    title="Global t-SNE of Multilingual Sentence Embeddings (Colored by Language)",

)
fig.update_traces(marker=dict(size=6, opacity=0.8))
fig.show()

In [ ]:
# Filter only Row 0
row_0_df = tsne_df[tsne_df["Row"] == 9]

# Plot
fig = px.scatter(
    row_0_df,
    x='x',
    y='y',
    color='Language',
    hover_data=['Language'],
    title='t-SNE Semantic Map for Row 0 (Same Sentence Across Languages)',

)

fig.update_traces(marker=dict(size=10, opacity=0.9))
fig.update_layout(showlegend=True)
fig.show()


### Cosine SImilarity on Original 768-D Embeddings


In [ ]:
similarity_matrices = []  # to hold similarity matrix per sentence

for idx, row in pm_ind_embeddings_df.iterrows():
    embeddings = np.array(row.tolist())  # shape: [num_languages, embedding_dim]
    sim_matrix = cosine_similarity(embeddings)  # shape: [num_langs x num_langs]
    similarity_matrices.append(sim_matrix)

In [ ]:
# Stack all matrices: shape becomes [num_rows, num_langs, num_langs]
similarity_stack = np.stack(similarity_matrices)

# Mean similarity matrix across all rows
avg_similarity_matrix = np.mean(similarity_stack, axis=0)


In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(avg_similarity_matrix, annot=True, fmt=".2f",
            xticklabels=pm_ind_embeddings_df.columns.tolist(),
            yticklabels=pm_ind_embeddings_df.columns.tolist(),
            cmap="YlGnBu")
plt.title("Average Cosine Similarity Matrix Across All Sentences")
plt.tight_layout()
plt.show()


### Cosine Similarity on t-SNE reduced embeddings - IndicBERT

In [ ]:
tsne_wide_df = (
    tsne_df
    .assign(coords=lambda df: df[['x', 'y']].values.tolist())
    .pivot(index='Row', columns='Language', values='coords')
)

In [ ]:
tsne_wide_df.head()

In [ ]:
similarity_matrices_tsne = []  # to hold similarity matrix per sentence

for idx, row in tsne_wide_df.iterrows():
    embeddings = np.array(row.tolist())
    sim_matrix = cosine_similarity(embeddings)
    similarity_matrices_tsne.append(sim_matrix)

In [ ]:
# Stack all matrices: shape becomes [num_rows, num_langs, num_langs]
similarity_stack_tsne = np.stack(similarity_matrices_tsne)

# Mean similarity matrix across all rows
avg_similarity_matrix_tsne = np.mean(similarity_stack_tsne, axis=0)


In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(avg_similarity_matrix_tsne, annot=True, fmt=".2f",
            xticklabels=tsne_wide_df.columns.tolist(),
            yticklabels=tsne_wide_df.columns.tolist(),
            cmap="YlGnBu")
plt.title("Average Cosine Similarity Matrix Across All Sentences with t-SNE reduced embeddings")
plt.tight_layout()
plt.show()